![JohnSnowLabs](https://sparknlp.org/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp/blob/SPARKNLP-1224-Implement-Dolphin-Document-Image-Parsing/examples/python/transformers/onnx/HuggingFace_ONNX_in_Spark_NLP_DolphinForDocumentParsing.ipynb)


# Import ONNX Dolphin models from HuggingFace 🤗 into Spark NLP 🚀

[Dolphin](https://huggingface.co/ByteDance/Dolphin-1.5) parses a document image into layout
elements in reading order: tables as HTML, formulas as LaTeX, text as markdown.

A few things to know before starting:

- ONNX support was introduced in `Spark NLP 5.0.0`.
- `DolphinForDocumentParsing` needs the **encoder + decoder + decoder_with_past** export
  (`image-to-text-with-past`). A cacheless decoder is not an option: cross-attention over the
  encoder's 784 states would be recomputed every step, roughly 500x the compute on a
  1000-token table.

### Which checkpoints work

| checkpoint | architecture | supported |
|---|---|---|
| `ByteDance/Dolphin-1.5` | `VisionEncoderDecoderModel` (donut-swin + mBart) | ✅ default, verified byte-exact |
| `ByteDance/Dolphin` | identical architecture and configs | ✅ verified byte-exact |
| `ByteDance/Dolphin-v2` | `Qwen2_5_VLForConditionalGeneration` | ❌ different architecture |

Dolphin 1.5 and 1.0 differ only in their weights. The annotator recognises the release from its
stage-1 output, so either export loads the same way; `MODEL_NAME` below picks one.

Dolphin-v2 is a 3B Qwen2.5-VL model with no encoder/decoder split and a chat template. It is
not loadable by this annotator and would need a separate one.


## 1. Export to ONNX


In [ ]:
# Colab preinstalls a large ML stack that fights this pipeline's pins. Three separate breakages,
# all headed off here:
#
# 1. transformers 4.47.1 pulls huggingface_hub below 1.0, which breaks Colab's much newer
#    `diffusers` (it imports `get_cached_repo_tree`, a hub 1.x API). optimum imports diffusers
#    opportunistically, so a broken-but-installed copy takes optimum-cli down with it. A
#    VisionEncoderDecoder export has nothing to do with diffusion.
# 2. torch and torchvision must move together. Upgrading torch alone leaves torchvision's compiled
#    extension bound to the old ABI -> "RuntimeError: operator torchvision::nms does not exist".
# 3. Newer torch defaults torch.onnx.export to the dynamo exporter, which rejects the legacy
#    `dynamic_axes` optimum 1.24.0 passes ("user-specified dim hint Dim.DYNAMIC, but tracing
#    inferred a static shape of 3 for inputs['pixel_values'].shape[1]") and silently bumps the
#    opset 14 -> 18. Spark NLP's loader reads this graph by name (`present.*` /
#    `past_key_values.*`, 40 in / 20 out), and those names were verified against the TorchScript
#    exporter -- so pin torch to a release that still defaults to it rather than take a graph
#    nobody has checked.
!pip uninstall -q -y diffusers
# 4. Pillow must be one coherent version. Left to `--upgrade` it ends up mixed -- ImageDraw from
#    one release, _typing from another -- which surfaces as
#    "ImportError: cannot import name '_Ink' from 'PIL._typing'" the moment transformers loads a
#    processor. --force-reinstall guarantees a single version on disk; the exact number matters
#    far less than the coherence.
!pip install -q --upgrade "transformers==4.47.1" "optimum[onnxruntime]==1.24.0" \
    "torch==2.6.0" "torchvision==0.21.0" onnxscript
!pip install -q --force-reinstall --no-cache-dir "pillow==11.1.0"


In [ ]:
MODEL_NAME = "ByteDance/Dolphin-1.5"
EXPORT_PATH = f"export_onnx/{MODEL_NAME}"

# Demo images come from ByteDance's own Dolphin repo (MIT licensed), pinned to the v1.5 release.
# Spark NLP's fixtures under src/test/resources/dolphin/ are byte-identical copies of most of
# these -- verified by sha256 -- but were renumbered on the way in: our page_1.jpeg is upstream's
# page_2.jpeg. Sourcing from upstream means this notebook works whether or not the Spark NLP
# branch has been pushed.
DEMO = "https://raw.githubusercontent.com/bytedance/Dolphin/80a3d85d58314a94fd6c741acf48b8fcb333240a/demo"


In [ ]:
!optimum-cli export onnx \
  --model {MODEL_NAME} \
  --task image-to-text-with-past \
  --opset 14 --device cpu --dtype fp32 --atol 1e-3 \
  {EXPORT_PATH}


Four graphs come out. Spark NLP uses the first three; `decoder_model_merged.onnx` is emitted
for free and is not needed, so we delete it rather than ship 1.2 GB of dead weight.

| file | approx size |
|---|---|
| `encoder_model.onnx` | 288 MB |
| `decoder_model.onnx` | 1.2 GB |
| `decoder_with_past_model.onnx` | 1.1 GB |
| `decoder_model_merged.onnx` | 1.2 GB (unused) |


In [ ]:
!ls -lh {EXPORT_PATH}


In [ ]:
import os, onnx

# Spark NLP reads this graph BY NAME, so a structurally different export would not fail loudly --
# it would decode subtly wrong. Check the shape here, while the export is still cheap to redo.
expected = ["encoder_model.onnx", "decoder_model.onnx", "decoder_with_past_model.onnx"]
missing = [f for f in expected if not os.path.exists(f"{EXPORT_PATH}/{f}")]
assert not missing, f"missing graphs: {missing}"

def io_names(fname):
    m = onnx.load(f"{EXPORT_PATH}/{fname}", load_external_data=False)
    return [i.name for i in m.graph.input], [o.name for o in m.graph.output]

enc_in, enc_out = io_names("encoder_model.onnx")
dec_in, dec_out = io_names("decoder_model.onnx")
past_in, past_out = io_names("decoder_with_past_model.onnx")

assert "pixel_values" in enc_in, enc_in
assert "last_hidden_state" in enc_out, enc_out
assert {"input_ids", "encoder_hidden_states"} <= set(dec_in), dec_in

present = [o for o in dec_out if o.startswith("present.")]
past = [i for i in past_in if i.startswith("past_key_values.")]
rolled = [o for o in past_out if o.startswith("present.")]

print(f"encoder    in {enc_in}  out {enc_out}")
print(f"decoder    present.* outputs: {len(present)}")
print(f"with_past  past_key_values.* inputs: {len(past)}   present.* outputs: {len(rolled)}")

# The 40/20 asymmetry the Scala inference loop is built on: prefill returns 40 tensors
# (10 layers x {encoder, decoder} x {key, value}); the with-past graph consumes all 40 but only
# rolls the 20 decoder ones forward, because encoder K/V never changes.
assert len(present) == 40, f"expected 40 present.* outputs, got {len(present)}"
assert len(past) == 40, f"expected 40 past_key_values.* inputs, got {len(past)}"
assert len(rolled) == 20, f"expected 20 present.*.decoder.* outputs, got {len(rolled)}"
assert not any("attention_mask" in n for n in dec_in), \
    "the decoder gained an attention-mask input; Spark NLP batches assume it has none"
print("\ngraph shape matches what DolphinForDocumentParsing expects")


## 2. Lay the assets out the way `loadSavedModel` expects

It reads `config.json`, `generation_config.json`, `preprocessor_config.json` and
`tokenizer.json` from an `assets/` subdirectory.


In [ ]:
!mkdir -p {EXPORT_PATH}/assets
!cp {EXPORT_PATH}/*.json {EXPORT_PATH}/assets/
!rm -f {EXPORT_PATH}/decoder_model_merged.onnx
!ls -l {EXPORT_PATH}/assets


## 3. Validate the export before trusting it

Dolphin's value rests on decoding page elements in batches, so it is worth confirming ONNX
Runtime and PyTorch agree at **more than one batch size**, and on **token ids** rather than
logit closeness — a logit-close export can still diverge mid-generation.


In [ ]:
import gc, itertools, torch
from PIL import Image
from transformers import AutoProcessor, VisionEncoderDecoderModel
from optimum.onnxruntime import ORTModelForVision2Seq

proc = AutoProcessor.from_pretrained(MODEL_NAME)
tok = proc.tokenizer
pt = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME).eval().float()
ort = ORTModelForVision2Seq.from_pretrained(EXPORT_PATH, use_io_binding=False)


In [ ]:
# the prompt ids Spark NLP asserts at model load; a mismatch here means the tokenizer changed
GOLDEN = {
    "layout": [0, 47928, 286, 7996, 1160, 299, 495, 5262, 36, 73920],
    "text":   [0, 17657, 3360, 301, 286, 1990, 36, 73920],
    "table":  [0, 47928, 286, 4345, 301, 286, 1990, 36, 73920],
    "formula": [0, 17657, 3914, 301, 286, 1990, 36, 73920],
    "code":   [0, 17657, 3355, 301, 286, 1990, 36, 73920],
}
PROMPTS = {
    "layout": "Parse the reading order of this document.",
    "text":   "Read text in the image.",
    "table":  "Parse the table in the image.",
    "formula": "Read formula in the image.",
    "code":   "Read code in the image.",
}
for key, prompt in PROMPTS.items():
    ids = tok(f"<s>{prompt} <Answer/>", add_special_tokens=False).input_ids
    assert ids == GOLDEN[key], (key, ids)
print("prompt ids OK")


In [ ]:
# Upstream's table.jpg is not the table_1.jpeg fixture in this repo, so its text will not match
# the parity references. That is fine here: this cell only checks that ONNX Runtime and PyTorch
# agree with each other on the same input.
!wget -q {DEMO}/element_imgs/table.jpg
import os
assert os.path.getsize("table.jpg") > 1024, "table.jpg did not download"

def greedy(model, pixel_values, prompt_ids, max_new=120, mask=True):
    # the exported decoder has no attention-mask input, so ORT will reject one
    kwargs = {"decoder_attention_mask": torch.ones_like(prompt_ids)} if mask else {}
    with torch.no_grad():
        return model.generate(
            pixel_values=pixel_values, decoder_input_ids=prompt_ids, **kwargs,
            min_length=1, max_length=prompt_ids.shape[1] + max_new,
            pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id,
            use_cache=True, bad_words_ids=[[tok.unk_token_id]],
            do_sample=False, num_beams=1)

image = Image.open("table.jpg").convert("RGB")

# BATCH SIZES ARE A MEMORY DIAL. Both models are resident here -- PyTorch fp32 weights plus three
# ONNX Runtime sessions, ~3.5 GB before a single token is generated. On top of that, prefill
# logits are [batch, prompt_len, 73921] floats: ~47 MB at batch 16 against ~12 MB at batch 4, and
# the KV cache scales the same way. Batch 16 OOMs a standard 12 GB Colab runtime.
#
# 1 and 4 are the sizes that matter anyway: 4 is DolphinForDocumentParsing's `elementBatchSize`
# default, and the point of the check is that ORT and PyTorch agree at MORE THAN ONE batch size,
# not at any particular one. Add 16 only on a high-RAM runtime.
BATCHES = [1, 4]
MAX_NEW = 40   # agreement diverges early if it is going to; 120 tokens just costs memory

for task, batch in itertools.product(["text", "table", "formula", "code", "layout"], BATCHES):
    pv = proc([image] * batch, return_tensors="pt").pixel_values.float()
    pid = torch.tensor([GOLDEN[task]] * batch)
    a = greedy(pt, pv, pid, max_new=MAX_NEW)
    b = greedy(ort, pv, pid, max_new=MAX_NEW, mask=False)
    same = a.shape == b.shape and bool((a == b).all())
    print(f"{task:7s} B={batch:<3d} {'OK' if same else 'MISMATCH'}")
    assert same, f"ORT and PyTorch diverged for task={task} batch={batch}"
    del pv, pid, a, b
    gc.collect()


### Take the export with you

The export is ~2.6 GB, too large for `files.download()`. Copy it to Drive instead — this is the
handoff point if you are doing the Spark NLP half on a local checkout.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/dolphin_onnx
!cp -r {EXPORT_PATH}/. /content/drive/MyDrive/dolphin_onnx/
!du -sh /content/drive/MyDrive/dolphin_onnx && ls /content/drive/MyDrive/dolphin_onnx

# Then, on the checkout:
#   DOLPHIN_ONNX_PATH=/path/to/dolphin_onnx sbt -mem 8192 "SlowTest/testOnly *Dolphin*"


## 4. Import into Spark NLP

> **This section needs a Spark NLP build that contains `DolphinForDocumentParsing`.**
> `sparknlp.start()` installs the released Spark NLP, which does not have this annotator yet, so
> `from sparknlp.annotator import DolphinForDocumentParsing` will fail on a stock Colab runtime.
> Until the annotator ships, either:
>
> 1. **Export here, test locally** (simplest): stop after section 3, download `EXPORT_PATH`, and
>    run the Spark NLP half on a checkout of the branch with
>    `DOLPHIN_ONNX_PATH=/path/to/export sbt "SlowTest/testOnly *Dolphin*"`; or
> 2. **Bring the branch build to Colab**: upload the assembled `python/lib/sparknlp.jar` plus the
>    branch's `python/sparknlp` package, and start Spark with
>    `SparkSession.builder.config("spark.jars", "sparknlp.jar")` instead of `sparknlp.start()`.


In [ ]:
!wget -q http://setup.johnsnowlabs.com/colab.sh -O - | bash


In [ ]:
import sparknlp

# Dolphin holds ~2.6 GB of fp32 weights in ONNX Runtime's *native* memory, which is outside
# the JVM heap. Budget for it via memoryOverhead, not executor memory.
spark = sparknlp.start()
spark.conf.set("spark.jsl.settings.onnx.intraOpNumThreads", "4")  # see the tuning note below


In [ ]:
from sparknlp.annotator import DolphinForDocumentParsing

dolphin = DolphinForDocumentParsing.loadSavedModel(EXPORT_PATH, spark) \
    .setInputCols(["image_assembler"]) \
    .setOutputCol("elements") \
    .setParsingMode("page")


In [ ]:
dolphin.write().overwrite().save(f"{EXPORT_PATH}_spark_nlp")
!rm -rf {EXPORT_PATH}
!ls -lh {EXPORT_PATH}_spark_nlp


## 5. Parse a page end to end

`page` mode runs both stages: layout analysis, then each element cropped and parsed,
batched by type.


In [ ]:
# upstream page_2.jpeg is byte-identical to this repo's page_1.jpeg fixture
!wget -q {DEMO}/page_imgs/page_2.jpeg -O page_1.jpeg

from sparknlp.base import ImageAssembler
from pyspark.ml import Pipeline

imageDF = spark.read.format("image").option("dropInvalid", True).load("page_1.jpeg")

pipeline = Pipeline(stages=[
    ImageAssembler().setInputCol("image").setOutputCol("image_assembler"),
    DolphinForDocumentParsing.load(f"{EXPORT_PATH}_spark_nlp")
        .setInputCols(["image_assembler"]).setOutputCol("elements"),
])

result = pipeline.fit(imageDF).transform(imageDF)
result.selectExpr("explode(elements) as e") \
      .selectExpr("e.metadata['readingOrder'] as order",
                  "e.metadata['dolphinLabel'] as label",
                  "e.metadata['elementType'] as type",
                  "e.result") \
      .orderBy("order").show(truncate=60)


Tables additionally carry `tableJson`, in the same shape `Reader2Table` produces, so a scanned
table drops into a pipeline that already consumes digital ones.


In [ ]:
tables = result.selectExpr("explode(elements) as e") \
               .filter("e.metadata['dolphinLabel'] = 'tab'")
for row in tables.collect():
    print(row["e"]["metadata"].get("tableJson", "(no table on this page)")[:400])


### Markdown instead of elements

`outputFormat="markdown"` emits one assembled document per page rather than one annotation
per element — the shape you want feeding a chunker and embeddings.


In [ ]:
md_model = DolphinForDocumentParsing.load(f"{EXPORT_PATH}_spark_nlp") \
    .setInputCols(["image_assembler"]).setOutputCol("elements") \
    .setOutputFormat("markdown")

pipeline = Pipeline(stages=[
    ImageAssembler().setInputCol("image").setOutputCol("image_assembler"), md_model])
out = pipeline.fit(imageDF).transform(imageDF).select("elements").collect()[0][0][0].result
print(out[:1500])


### Layout only

`layout` mode runs stage 1 alone: regions, labels and reading order, without paying to
transcribe anything. Useful for finding every table across a large corpus cheaply.


In [ ]:
layout_model = DolphinForDocumentParsing.load(f"{EXPORT_PATH}_spark_nlp") \
    .setInputCols(["image_assembler"]).setOutputCol("elements") \
    .setParsingMode("layout")

pipeline = Pipeline(stages=[
    ImageAssembler().setInputCol("image").setOutputCol("image_assembler"), layout_model])
pipeline.fit(imageDF).transform(imageDF) \
    .selectExpr("explode(elements) as e") \
    .selectExpr("e.metadata['dolphinLabel'] as label",
                "e.metadata['bbox'] as bbox",
                "e.metadata['readingGroup'] as col") \
    .show(30, truncate=False)


## 6. Tuning notes

**Memory.** The fp32 model is ~2.6 GB of ONNX Runtime session memory, held *natively* — outside
the JVM heap, so `spark.executor.memory` does not cover it. Budget
`spark.executor.memoryOverhead` accordingly, or publish/use an int8 export (~850 MB).

**`elementBatchSize`** defaults to 4. Measured throughput at 4 matched or beat 16 on every
task, while the KV cache at batch 16 with a 4096-token window needs over 10 GB. Raising it is
usually a loss on both counts.

**`spark.jsl.settings.onnx.intraOpNumThreads`** is worth setting. On a 10-core machine
(4 performance + 6 efficiency) the optimum was 4 — matching physical performance cores, not
logical processors — for about 6%. Going above that count was actively slower. Measure on your
own hardware; `DolphinThreadBenchmark` in the test sources does exactly this.
